In [ ]:
# ==============================================================================
# FASE 1 Y 2: INGESTA, PREPROCESAMIENTO Y MOTOR PREDICTIVO (XGBoost)
# ==============================================================================
import pandas as pd
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.over_sampling import SMOTE

# 1. Ingesta de datos ( desde la nube)
# En repositorio público de GitHub 
#  ruta local.
url_dataset = "https://raw.githubusercontent.com/yunquel1/heart-failure-xgboost-shap/main/heart.csv"
df = pd.read_csv(url_dataset)

# Separación de características (X) y variable predictiva (y)
X = df.drop('target', axis=1)
y = df['target']

# Balanceo de clases con SMOTE 
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# División del dataset (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

# 2. Configuración y Entrenamiento del Modelo XGBoost
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=1.2, 
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Verificación de la métrica (Recall > 85%)
y_pred = xgb_model.predict(X_test)
recall = recall_score(y_test, y_pred)
print(f"Sensibilidad (Recall) del Modelo XGBoost: {recall * 100:.2f}%")

# ==============================================================================
# FASE 3: MÓDULO DE EXPLICABILIDAD CLÍNICA (SHAP)
# ==============================================================================
shap.initjs()
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_train)